# Min-K% Prob — Master Notebook

This notebook orchestrates the full Min-K% Prob replication pipeline.
Each module calls a script in `scripts/` which in turn calls `src/`.
No logic is written inline here — the notebook is a runner, not a codebase.

**Isolation contract:**
- `src/` — pure Python logic (imported by scripts, tested by pytest)
- `scripts/` — orchestration, CLI entry points, Drive I/O
- This notebook — runs scripts, displays outputs, writes nothing to `src/`

---

## [0] Runtime and GPU Check

Verifies the Colab runtime has GPU access and confirms the Python environment before any module runs.

In [ ]:
import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
else:
    print('GPU     : None — switch Runtime > Change runtime type > GPU')

## [1] Drive Mount and Repository Setup

Mounts the shared Google Drive folder and clones the repository onto the Colab instance.
The `DRIVE` variable is set here and exported as an environment variable so all downstream scripts resolve their output paths without hardcoding.

> **First-time setup:** create the shared folder in Drive and share it with all team members before running this cell.

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Shared Drive folder (all members use the same path once shared)
DRIVE = '/content/drive/MyDrive/min-k-project-group2'
os.environ['DRIVE'] = DRIVE
Path(DRIVE).mkdir(parents=True, exist_ok=True)
print(f'Drive mounted at : {DRIVE}')

# Set environment variable for downstream scripts
%env DRIVE=/content/drive/MyDrive/min-k-project-group2

# Repo clone / pull
REPO_URL    = 'https://github.com/dazzlear/ai-final-project.git'
BRANCH      = 'paraphrase-experiment' 
PROJECT_DIR = '/content/ai-final-project'

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git checkout {BRANCH}
!git pull origin {BRANCH}
print(f'Repo ready at : {PROJECT_DIR}')


## [2] Configuration

**This is the only cell that should be edited between runs.**
All scripts read these variables via CLI arguments or the `DRIVE` environment variable.

| Variable | Purpose |
|---|---|
| `LENGTHS` | WikiMIA length buckets to process |
| `DEFAULT_MODEL` | Model used for smoke test and full runs |
| `SAMPLE_SIZE` | Rows per sample CSV (balanced: half per label) |
| `DRIVE` | Root of the shared Drive output folder |

In [ ]:
# Edit this cell only

LENGTHS       = [32, 64, 128, 256]   # subset e.g. [64] for a quick test
DEFAULT_MODEL = 'EleutherAI/pythia-410m'
SAMPLE_SIZE   = 10

MODELS = {
    'pythia410m' : 'EleutherAI/pythia-410m',    # smoke-test model only
    'pythia2.8b' : 'EleutherAI/pythia-2.8b',
    'gptneo1.3b' : 'EleutherAI/gpt-neo-1.3B',
    'opt1.3b'    : 'facebook/opt-1.3b',
}
SMOKE_MODEL = 'pythia410m'

# Derived paths -- do not edit
DRIVE_01 = f'{DRIVE}/01_dataset'
DRIVE_02 = f'{DRIVE}/02_model_loading'
DRIVE_03 = f'{DRIVE}/03_mink_scores'
DRIVE_04 = f'{DRIVE}/04_baseline_scores'
DRIVE_05 = f'{DRIVE}/05_evaluation'

print('Config set:')
print(f'  LENGTHS       = {LENGTHS}')
print(f'  DEFAULT_MODEL = {DEFAULT_MODEL}')
print(f'  DRIVE_01      = {DRIVE_01}')

## [3] Install Dependencies

Installs all required packages. This cell is idempotent — safe to re-run.

In [ ]:
%pip install -q datasets transformers torch scikit-learn pandas numpy matplotlib tqdm sentencepiece protobuf tiktoken


## [4] Module 01 — Dataset Loading, Preparation, and Paraphrased Dataset Generation

Calls `scripts/run_01_dataset.py` to load, validate, and save all configured WikiMIA length splits from Hugging Face.  
It also runs Module 1B to generate the paraphrased WikiMIA length64 dataset.

**What this section produces** (written to `{DRIVE}/01_dataset/`):

| File | Description |
|---|---|
| `wikimia_length{N}_processed.csv` | Full dataset for each length |
| `wikimia_length{N}_sample.csv` | 10-row balanced smoke-test sample |
| `wikimia_length64_paraphrased.csv` | Paraphrased version of the length64 dataset for the original vs paraphrased experiment |
| `summaries/dataset_summary_len{N}.txt` | Quality report per length |
| `summaries/dataset_summary_all.csv` | Combined summary across all lengths |

The script validates each split for missing values, empty texts, and label balance before saving.  
The paraphrased file is saved directly to Google Drive through `DRIVE_01`, following the same Drive output pattern as the other Module 1 files.


In [ ]:
# Build the --lengths argument string from the config cell
_lengths_arg = ' '.join(str(l) for l in LENGTHS)

!python scripts/run_01_dataset.py \
    --lengths {_lengths_arg} \
    --output_dir {DRIVE_01} \
    --sample_size {SAMPLE_SIZE} \
    --make_paraphrase \
    --paraphrase_length 64 \
    --paraphrase_limit 0 \
    --paraphrase_model Vamsi/T5_Paraphrase_Paws


### 4.1 Module 1B Paraphrased Dataset Check

Verifies that the paraphrased WikiMIA length64 dataset was saved to the same Google Drive output folder used by Module 1: `{DRIVE}/01_dataset/`.


In [ ]:
from pathlib import Path
import pandas as pd

paraphrase_path = Path(DRIVE_01) / "wikimia_length64_paraphrased.csv"

print("Checking paraphrased dataset in Google Drive...")
print(f"Path: {paraphrase_path}")
print(f"Exists: {paraphrase_path.exists()}")

if paraphrase_path.exists():
    df_para = pd.read_csv(paraphrase_path)

    print(f"Rows: {len(df_para)}")
    print(f"Columns: {df_para.columns.tolist()}")

    required_cols = ["text_id", "label", "original_text", "text", "is_paraphrased"]

    for col in required_cols:
        print(f"{'✅' if col in df_para.columns else '❌'} {col}")

    display(df_para[["text_id", "label", "original_text", "text"]].head(3))
else:
    print("❌ Paraphrased dataset was not found in DRIVE_01. Run Module 1 first.")


### 4.2 Output Preview

Displays the first few rows and label distribution for each processed length split.
Member (label=1) and non-member (label=0) counts should be equal for all lengths.


In [ ]:
from pathlib import Path
import pandas as pd

for length in LENGTHS:
    path = Path(DRIVE_01) / f'wikimia_length{length}_processed.csv'
    if path.exists():
        df = pd.read_csv(path)
        print(f'\n{"="*55}')
        print(f'  WikiMIA_length{length}  --  {len(df)} rows')
        print(f'{"="*55}')
        display(df.head(3))
        vc = df['label'].value_counts().rename({0: 'non-member (0)', 1: 'member (1)'})
        print(vc.to_string())
    else:
        print(f'[MISSING] {path}')

### 4.3 Module 01 Sanity Check

Verifies that every expected output file exists and that each dataset meets the minimum quality bar before proceeding to Module 02.

**Pass criteria (all must be green):**
- All processed CSVs present for every configured length
- All sample CSVs present
- No missing text or label values
- Labels are balanced (label_0 == label_1)
- No empty text rows


In [ ]:
from pathlib import Path
import pandas as pd

PASS = True

for length in LENGTHS:
    processed = Path(DRIVE_01) / f'wikimia_length{length}_processed.csv'
    sample    = Path(DRIVE_01) / f'wikimia_length{length}_sample.csv'
    summary   = Path(DRIVE_01) / 'summaries' / f'dataset_summary_len{length}.txt'

    checks = {
        f'processed CSV exists (len{length})': processed.exists(),
        f'sample CSV exists    (len{length})': sample.exists(),
        f'summary txt exists   (len{length})': summary.exists(),
    }

    if processed.exists():
        df = pd.read_csv(processed)
        checks[f'no missing text   (len{length})'] = df['text'].isna().sum() == 0
        checks[f'no missing labels (len{length})'] = df['label'].isna().sum() == 0
        checks[f'no empty texts    (len{length})'] = (df['text'].str.strip() == '').sum() == 0

        label_0 = int((df['label'] == 0).sum())
        label_1 = int((df['label'] == 1).sum())
        larger  = max(label_0, label_1)
        smaller = min(label_0, label_1)
        ratio   = smaller / larger

        if ratio < 0.90:
            # Imbalance is expected for longer lengths due to upstream filtering.
            # Logged as a warning — does not fail the pipeline.
            print(f'  ⚠️   label imbalance (len{length}): {label_0} vs {label_1} '
                  f'(ratio {ratio:.2f}) — known upstream dataset property')
        else:
            checks[f'labels balanced (len{length}) [{label_0} vs {label_1}]'] = True

    for name, ok in checks.items():
        icon = '✅' if ok else '❌'
        print(f'  {icon}  {name}')
        if not ok:
            PASS = False

master = Path(DRIVE_01) / 'summaries' / 'dataset_summary_all.csv'
icon = '✅' if master.exists() else '❌'
print(f'  {icon}  master summary CSV exists')
if not master.exists():
    PASS = False

print()
if PASS:
    print('✅  Module 01 PASSED — proceed to Module 02.')
else:
    print('❌  Module 01 FAILED — fix the issues above before running Module 02.')

## [5] Module 02 — Model Loading and Verification

Calls `scripts/run_02_model_loading.py` to load each model in the `MODELS`
config, run a diagnostics check, and save a verification report to Drive.
No log-probabilities are computed here — that is Module 03.

**What this section produces** (written to `{DRIVE}/02_model_loading/`):

| File | Description |
|---|---|
| `model_verification_report.csv` | One row per model: architecture, param count, VRAM, vocab size, roundtrip and forward-pass results |

The script loads each model, captures diagnostics, runs a tokenization
roundtrip and a single forward pass to confirm the model is functional,
then unloads it before moving to the next. All four models are verified
in sequence.

> **Before running:** confirm Module 01 passed (all ✅ in §4.2).

In [ ]:
# ── Module 02 run ─────────────────────────────────────────────────────────────
# Verifies all models in MODELS can load and produce logits.
# Outputs: model_verification_report.csv in DRIVE_02.

_models_arg = ' '.join(MODELS.values())

!python scripts/run_02_model_loading.py \
    --models {_models_arg} \
    --output_dir {DRIVE_02}

### 5.1 Output Preview

Displays the verification report table. Every model should show
`roundtrip_ok = True` and `forward_pass_ok = True` before proceeding.
VRAM readings confirm the model fit in GPU memory without OOM.

In [ ]:
from pathlib import Path
import pandas as pd

report_path = Path(DRIVE_02) / 'model_verification_report.csv'

if report_path.exists():
    df_report = pd.read_csv(report_path)
    display(df_report[[
        'model_key', 'architecture', 'parameters_M',
        'vram_mb', 'vocab_size', 'roundtrip_ok', 'forward_pass_ok'
    ]])

    failed = df_report[
        ~df_report['roundtrip_ok'] | ~df_report['forward_pass_ok']
    ]
    if len(failed) == 0:
        print('\nAll models passed verification ✅')
    else:
        print('\n⚠️  Failed models:')
        print(failed[['model_key', 'roundtrip_ok', 'forward_pass_ok']].to_string(index=False))
else:
    print(f'[MISSING] {report_path}')
    print('Re-run the cell above.')

### 5.2 Module 02 Sanity Check

Verifies the verification report exists and that every configured model
passed both checks before proceeding to Module 03.

**Pass criteria (all must be green):**
- `model_verification_report.csv` present
- One row per model in `MODELS`
- All rows: `roundtrip_ok = True`
- All rows: `forward_pass_ok = True`

In [ ]:
from pathlib import Path
import pandas as pd

PASS = True
report_path = Path(DRIVE_02) / 'model_verification_report.csv'

# ── Report file exists ────────────────────────────────────────────────────────
report_ok = report_path.exists()
print(f'  {"✅" if report_ok else "❌"}  model_verification_report.csv exists')
if not report_ok:
    PASS = False
else:
    df_r = pd.read_csv(report_path)

    # ── Row count matches MODELS ──────────────────────────────────────────────
    expected_n = len(MODELS)
    found_n    = len(df_r)
    count_ok   = found_n == expected_n
    print(f'  {"✅" if count_ok else "❌"}  row count: {found_n} / {expected_n} expected')
    if not count_ok:
        PASS = False

    # ── Per-model checks ──────────────────────────────────────────────────────
    print()
    for _, row in df_r.iterrows():
        rt_ok = bool(row['roundtrip_ok'])
        fp_ok = bool(row['forward_pass_ok'])
        icon  = '✅' if (rt_ok and fp_ok) else '❌'
        print(f'  {icon}  {row["model_key"]:15s}  '
              f'roundtrip={rt_ok}  forward_pass={fp_ok}  '
              f'{row["parameters_M"]:.0f}M params  {row["vram_mb"]:.0f} MB VRAM')
        if not (rt_ok and fp_ok):
            PASS = False

print()
if PASS:
    print('✅  Module 02 PASSED — proceed to Module 03.')
else:
    print('❌  Module 02 FAILED — fix the issues above before running Module 03.')

## [0] Runtime and GPU Check

Verifies the Colab runtime has GPU access and confirms the Python environment before any module runs.

In [ ]:
import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
else:
    print('GPU     : None — switch Runtime > Change runtime type > GPU')

## [1] Drive Mount and Repository Setup

Mounts the shared Google Drive folder and clones the repository onto the Colab instance.
The `DRIVE` variable is set here and exported as an environment variable so all downstream scripts resolve their output paths without hardcoding.

> **First-time setup:** create the shared folder in Drive and share it with all team members before running this cell.

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Shared Drive folder (all members use the same path once shared)
DRIVE = '/content/drive/MyDrive/min-k-project-group2'
os.environ['DRIVE'] = DRIVE
Path(DRIVE).mkdir(parents=True, exist_ok=True)
print(f'Drive mounted at : {DRIVE}')

# Set environment variable for downstream scripts
%env DRIVE=/content/drive/MyDrive/min-k-project-group2

# Repo clone / pull
REPO_URL    = 'https://github.com/dazzlear/ai-final-project.git'
BRANCH      = 'codebase-cleanup'
PROJECT_DIR = '/content/ai-final-project'

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git checkout {BRANCH}
!git pull origin {BRANCH}
print(f'Repo ready at : {PROJECT_DIR}')

## [2] Configuration

**This is the only cell that should be edited between runs.**
All scripts read these variables via CLI arguments or the `DRIVE` environment variable.

| Variable | Purpose |
|---|---|
| `LENGTHS` | WikiMIA length buckets to process |
| `DEFAULT_MODEL` | Model used for smoke test and full runs |
| `SAMPLE_SIZE` | Rows per sample CSV (balanced: half per label) |
| `DRIVE` | Root of the shared Drive output folder |

In [ ]:
# Edit this cell only

LENGTHS       = [32, 64, 128, 256]   # subset e.g. [64] for a quick test
DEFAULT_MODEL = 'EleutherAI/pythia-410m'
SAMPLE_SIZE   = 10

MODELS = {
    'pythia410m' : 'EleutherAI/pythia-410m',    # smoke-test model only
    'pythia2.8b' : 'EleutherAI/pythia-2.8b',
    'gptneo1.3b' : 'EleutherAI/gpt-neo-1.3B',
    'opt1.3b'    : 'facebook/opt-1.3b',
}
SMOKE_MODEL = 'pythia410m' 

# Derived paths -- do not edit
DRIVE_01 = f'{DRIVE}/01_dataset'
DRIVE_02 = f'{DRIVE}/02_model_loading'
DRIVE_03 = f'{DRIVE}/03_mink_scores'
DRIVE_04 = f'{DRIVE}/04_baseline_scores'
DRIVE_05 = f'{DRIVE}/05_evaluation'

print('Config set:')
print(f'  LENGTHS       = {LENGTHS}')
print(f'  DEFAULT_MODEL = {DEFAULT_MODEL}')
print(f'  DRIVE_01      = {DRIVE_01}')

## [3] Install Dependencies

Installs all required packages. This cell is idempotent — safe to re-run.

In [ ]:
!pip install -q datasets transformers torch scikit-learn pandas numpy matplotlib tqdm sentencepiece protobuf tiktoken

## [4] Module 01 — Dataset Loading and Preparation

Calls `scripts/run_01_dataset.py` to load, validate, and save all configured WikiMIA length splits from Hugging Face.

**What this section produces** (written to `{DRIVE}/01_dataset/`):

| File | Description |
|---|---|
| `wikimia_length{N}_processed.csv` | Full dataset for each length (used by Module 02) |
| `wikimia_length{N}_sample.csv` | 10-row balanced smoke-test sample |
| `summaries/dataset_summary_len{N}.txt` | Quality report per length |
| `summaries/dataset_summary_all.csv` | Combined summary across all lengths |

The script validates each split for missing values, empty texts, and label balance before saving.

In [ ]:
# Build the --lengths argument string from the config cell
_lengths_arg = ' '.join(str(l) for l in LENGTHS)

!python scripts/run_01_dataset.py \
    --lengths {_lengths_arg} \
    --output_dir {DRIVE_01} \
    --sample_size {SAMPLE_SIZE} \
    --make_paraphrase \
    --paraphrase_length 64 \
    --paraphrase_limit 0 \
    --paraphrase_model Vamsi/T5_Paraphrase_Paws

### 4.1 Output Preview

Displays the first few rows and label distribution for each processed length split.
Member (label=1) and non-member (label=0) counts should be equal for all lengths.

In [ ]:
from pathlib import Path
import pandas as pd

for length in LENGTHS:
    path = Path(DRIVE_01) / f'wikimia_length{length}_processed.csv'
    if path.exists():
        df = pd.read_csv(path)
        print(f'\n{"="*55}')
        print(f'  WikiMIA_length{length}  --  {len(df)} rows')
        print(f'{"="*55}')
        display(df.head(3))
        vc = df['label'].value_counts().rename({0: 'non-member (0)', 1: 'member (1)'})
        print(vc.to_string())
    else:
        print(f'[MISSING] {path}')

### 4.2 Module 01 Sanity Check

Verifies that every expected output file exists and that each dataset meets the minimum quality bar before proceeding to Module 02.

**Pass criteria (all must be green):**
- All processed CSVs present for every configured length
- All sample CSVs present
- No missing text or label values
- Labels are balanced (label_0 == label_1)
- No empty text rows

In [ ]:
from pathlib import Path
import pandas as pd

PASS = True

for length in LENGTHS:
    processed = Path(DRIVE_01) / f'wikimia_length{length}_processed.csv'
    sample    = Path(DRIVE_01) / f'wikimia_length{length}_sample.csv'
    summary   = Path(DRIVE_01) / 'summaries' / f'dataset_summary_len{length}.txt'

    checks = {
        f'processed CSV exists (len{length})': processed.exists(),
        f'sample CSV exists    (len{length})': sample.exists(),
        f'summary txt exists   (len{length})': summary.exists(),
    }

    if processed.exists():
        df = pd.read_csv(processed)
        checks[f'no missing text   (len{length})'] = df['text'].isna().sum() == 0
        checks[f'no missing labels (len{length})'] = df['label'].isna().sum() == 0
        checks[f'no empty texts    (len{length})'] = (df['text'].str.strip() == '').sum() == 0

        label_0 = int((df['label'] == 0).sum())
        label_1 = int((df['label'] == 1).sum())
        larger  = max(label_0, label_1)
        smaller = min(label_0, label_1)
        ratio   = smaller / larger

        if ratio < 0.90:
            # Imbalance is expected for longer lengths due to upstream filtering.
            # Logged as a warning — does not fail the pipeline.
            print(f'  ⚠️   label imbalance (len{length}): {label_0} vs {label_1} '
                  f'(ratio {ratio:.2f}) — known upstream dataset property')
        else:
            checks[f'labels balanced (len{length}) [{label_0} vs {label_1}]'] = True

    for name, ok in checks.items():
        icon = '✅' if ok else '❌'
        print(f'  {icon}  {name}')
        if not ok:
            PASS = False

master = Path(DRIVE_01) / 'summaries' / 'dataset_summary_all.csv'
icon = '✅' if master.exists() else '❌'
print(f'  {icon}  master summary CSV exists')
if not master.exists():
    PASS = False

print()
if PASS:
    print('✅  Module 01 PASSED — proceed to Module 02.')
else:
    print('❌  Module 01 FAILED — fix the issues above before running Module 02.')

## [5] Module 02 — Model Loading and Verification

Calls `scripts/run_02_model_loading.py` to load each model in the `MODELS`
config, run a diagnostics check, and save a verification report to Drive.
No log-probabilities are computed here — that is Module 03.

**What this section produces** (written to `{DRIVE}/02_model_loading/`):

| File | Description |
|---|---|
| `model_verification_report.csv` | One row per model: architecture, param count, VRAM, vocab size, roundtrip and forward-pass results |

The script loads each model, captures diagnostics, runs a tokenization
roundtrip and a single forward pass to confirm the model is functional,
then unloads it before moving to the next. All four models are verified
in sequence.

> **Before running:** confirm Module 01 passed (all ✅ in §4.2).

In [ ]:
# ── Module 02 run ─────────────────────────────────────────────────────────────
# Verifies all models in MODELS can load and produce logits.
# Outputs: model_verification_report.csv in DRIVE_02.

_models_arg = ' '.join(MODELS.values())

!python scripts/run_02_model_loading.py \
    --models {_models_arg} \
    --output_dir {DRIVE_02}

### 5.1 Output Preview

Displays the verification report table. Every model should show
`roundtrip_ok = True` and `forward_pass_ok = True` before proceeding.
VRAM readings confirm the model fit in GPU memory without OOM.

In [ ]:
from pathlib import Path
import pandas as pd

report_path = Path(DRIVE_02) / 'model_verification_report.csv'

if report_path.exists():
    df_report = pd.read_csv(report_path)
    display(df_report[[
        'model_key', 'architecture', 'parameters_M',
        'vram_mb', 'vocab_size', 'roundtrip_ok', 'forward_pass_ok'
    ]])

    failed = df_report[
        ~df_report['roundtrip_ok'] | ~df_report['forward_pass_ok']
    ]
    if len(failed) == 0:
        print('\nAll models passed verification ✅')
    else:
        print('\n⚠️  Failed models:')
        print(failed[['model_key', 'roundtrip_ok', 'forward_pass_ok']].to_string(index=False))
else:
    print(f'[MISSING] {report_path}')
    print('Re-run the cell above.')

### 5.2 Module 02 Sanity Check

Verifies the verification report exists and that every configured model
passed both checks before proceeding to Module 03.

**Pass criteria (all must be green):**
- `model_verification_report.csv` present
- One row per model in `MODELS`
- All rows: `roundtrip_ok = True`
- All rows: `forward_pass_ok = True`

In [ ]:
from pathlib import Path
import pandas as pd

PASS = True
report_path = Path(DRIVE_02) / 'model_verification_report.csv'

# ── Report file exists ────────────────────────────────────────────────────────
report_ok = report_path.exists()
print(f'  {"✅" if report_ok else "❌"}  model_verification_report.csv exists')
if not report_ok:
    PASS = False
else:
    df_r = pd.read_csv(report_path)

    # ── Row count matches MODELS ──────────────────────────────────────────────
    expected_n = len(MODELS)
    found_n    = len(df_r)
    count_ok   = found_n == expected_n
    print(f'  {"✅" if count_ok else "❌"}  row count: {found_n} / {expected_n} expected')
    if not count_ok:
        PASS = False

    # ── Per-model checks ──────────────────────────────────────────────────────
    print()
    for _, row in df_r.iterrows():
        rt_ok = bool(row['roundtrip_ok'])
        fp_ok = bool(row['forward_pass_ok'])
        icon  = '✅' if (rt_ok and fp_ok) else '❌'
        print(f'  {icon}  {row["model_key"]:15s}  '
              f'roundtrip={rt_ok}  forward_pass={fp_ok}  '
              f'{row["parameters_M"]:.0f}M params  {row["vram_mb"]:.0f} MB VRAM')
        if not (rt_ok and fp_ok):
            PASS = False

print()
if PASS:
    print('✅  Module 02 PASSED — proceed to Module 03.')
else:
    print('❌  Module 02 FAILED — fix the issues above before running Module 03.')